# Relevance Judgement
The process of determining how well a document (or passage) answers a given query. In Question-Answering(QA) tasks, this means ranking passages by how likely they contain the answer to a question.

## Task Overview
**Goal**: Given a question and a set of passages, score each passage for relevance to the question.

**Dataset**: [The Stanford Question Answering Dataset(SQuAD)](https://huggingface.co/datasets/rajpurkar/squad) - provides question-context pairs

**Approach**: Encode both question and document using a language model, then compare their representations.

### Two Approaches:
- Early Interaction
- Late Interation

In this, we will discuss about late interaction.

### Late interaction:
- Late interaction refers to comparing query and document representations *after* they have been independently encoded.
- We use token-level embeddings and a similarity operator to score relevance.


### [CLS] token
- [CLS] is a special token added at the start of every input sequence in BERT-like models.
- Its embedding is often used as a summary representation of the entire sequence.

In [1]:
# Importing necessary libraries
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import torch
import numpy as np

In [2]:
# Loading the dataset (First 20 examples from the validation set)
squad = load_dataset('squad', split='validation[:20]')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Display the first example
example = squad[0]
print("Question:", example['question'])
print("Context:", example['context'][:300] + "...")
print("Answer(s):", example['answers'])

Question: Which NFL team represented the AFC at Super Bowl 50?
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super B...
Answer(s): {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}


In [4]:
# Load Encoder Model & Tokenizer
model_name = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [5]:
# Encode Queries and Documents ([CLS] Embeddings)
def get_cls_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] token is the first token
    cls_emb = outputs.last_hidden_state[:, 0, :].squeeze(0)
    return cls_emb

# Example: encode first question and context
q_text = squad[0]['question']
d_text = squad[0]['context']
q_emb = get_cls_embedding(q_text)
d_emb = get_cls_embedding(d_text)
print('Query CLS shape:', q_emb.shape)
print('Document CLS shape:', d_emb.shape)

Query CLS shape: torch.Size([384])
Document CLS shape: torch.Size([384])


In [6]:
# Compute Relevance Score (Dot Product)
def relevance_score(q_emb, d_emb):
    # Normalize for cosine similarity
    q_norm = q_emb / q_emb.norm()
    d_norm = d_emb / d_emb.norm()
    score = torch.dot(q_norm, d_norm).item()
    return score

score = relevance_score(q_emb, d_emb)
print(f'Relevance score: {score:.4f}')

Relevance score: 0.8642


In [7]:
# Batch Scoring: Rank Contexts for a Query
q_text = squad[0]['question']
print("Question:", q_text)

# Encode query once
q_emb = get_cls_embedding(q_text)

scores = []
for i in range(len(squad)):
    d_text = squad[i]['context']
    d_emb = get_cls_embedding(d_text)
    score = relevance_score(q_emb, d_emb)
    scores.append((i, score))

# Sort by score descending
scores.sort(key=lambda x: x[1], reverse=True)
print(scores)
print('Top 3 most relevant contexts:')
for idx, score in scores[:3]:
    print(f'Index: {idx}, Score: {score:.4f}')
    print('Context:', squad[idx]['context'][:500], '...\n')

Question: Which NFL team represented the AFC at Super Bowl 50?
[(0, 0.8642380237579346), (1, 0.8642380237579346), (2, 0.8642380237579346), (3, 0.8642380237579346), (4, 0.8642380237579346), (5, 0.8642380237579346), (6, 0.8642380237579346), (7, 0.8642380237579346), (8, 0.8642380237579346), (9, 0.8642380237579346), (10, 0.8642380237579346), (11, 0.8642380237579346), (12, 0.8642380237579346), (13, 0.8642380237579346), (14, 0.8642380237579346), (15, 0.8642380237579346), (16, 0.8642380237579346), (17, 0.8642380237579346), (18, 0.8642380237579346), (19, 0.8642380237579346)]
Top 3 most relevant contexts:
Index: 0, Score: 0.8642
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's S

# Advantages:

- **Efficiency**: Document embeddings can be precomputed and stored, enabling fast retrieval.
- **Scalability**: Suitable for large collections, as only query encoding is needed at search time.
- **Fine-grained Matching**: Token-level comparison (e.g., MaxSim) captures nuanced relevance.
- **Interpretability**: Can inspect which tokens matched best.

# Disadvantages:
- **Storage Cost**: Need to store multiple embeddings per document (unless pooling or quantization is used).
- **Slightly Lower Expressiveness**: Compared to early interaction (cross-encoder) models, which allow richer token-to-token attention.
- **Complexity**: More complex retrieval infrastructure required for multi-vector search.

# References
- ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT : https://github.com/stanford-futuredata/ColBERT

- An Overview of Late Interaction Retrieval Models: https://weaviate.io/blog/late-interaction-overview

- Late Interaction & ColBERT: https://www.lancedb.com/documentation/studies/late-interaction-colbert.html

- Flexible Training and Retrieval for Late Interaction Models: https://arxiv.org/html/2508.03555v1

- ColBERT: A Token-Level Embedding and Ranking Model: https://zilliz.com/learn/explore-colbert-token-level-embedding-and-ranking-model-for-similarity-search